In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model


c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "gpt2"
OUTPUT_DIR = "./lora-gpt2-demo"

BATCH_SIZE = 2
EPOCHS = 3
LR = 2e-4
MAX_LENGTH = 128

# Load tokenizer & base model (FROZEN)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

# Freeze base model
for param in model.parameters():
    param.requires_grad = False


`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
# LoRA configuration

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],   # QKV projection ; Wc_attn​=[WQ ​​WK​ ​WV​​]
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [4]:
texts = [
    "The patient presents with chest pain and elevated troponin levels.",
    "MRI imaging shows abnormalities in the frontal cortex.",
    "The transformer architecture relies heavily on self-attention.",
    "LoRA allows efficient fine-tuning of large language models.",
    "Backpropagation updates only low-rank matrices in LoRA."
]

dataset = Dataset.from_dict({"text": texts})


In [6]:
def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_ds = dataset.map(tokenize, batched=True, remove_columns=["text"])


Map: 100%|██████████| 5/5 [00:00<00:00, 87.57 examples/s]


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=1,
    save_strategy="epoch",
    fp16=False,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


In [ ]:
# Save only LoRA adapters

model.save_pretrained("./lora-adapter")
tokenizer.save_pretrained("./lora-adapter")
